# Steam Gaming Trends — 03: Baseline Sentiment & Parquet Export

Applies VADER to a 100k review sample, compares against ground-truth `is_recommended` labels,  
and serialises the enriched dataset to `outputs/processed_reviews.parquet` for P17.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.')))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from src.sentiment import add_vader

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

DATA    = Path('data')
FIG     = Path('outputs/figures')
OUTPUTS = Path('outputs')
FIG.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

## 1. Load Review Sample

We join a 100k review sample with `games.csv` to attach game names.

In [ ]:
recs = pd.read_csv(DATA / 'recommendations.csv', nrows=100_000, low_memory=False)
games = pd.read_csv(DATA / 'games.csv', usecols=['app_id', 'name'], low_memory=False)

# join on app_id (recommendations) / app_id (games)
df = recs.merge(games, on='app_id', how='left')
df = df.rename(columns={'name': 'app_name', 'review': 'review_text'})
df = df[['app_name', 'review_text', 'is_recommended']].rename(
    columns={'is_recommended': 'recommended'}
)
df = df.dropna(subset=['review_text']).reset_index(drop=True)
print(f'Working sample: {len(df):,} reviews')

## 2. Apply VADER Sentiment

In [ ]:
df = add_vader(df, text_col='review_text')
df[['review_text', 'recommended', 'vader_compound', 'vader_polarity']].head(5)

## 3. VADER vs Ground Truth — 20-Review Spot Check

In [ ]:
spot = df.sample(20, random_state=42)[['review_text', 'recommended', 'vader_polarity', 'vader_compound']]
spot['ground_truth'] = spot['recommended'].map({True: 'positive', False: 'negative'})
spot['match'] = spot['vader_polarity'] == spot['ground_truth']
print(f'Spot-check agreement: {spot["match"].mean():.0%}')
spot[['review_text', 'ground_truth', 'vader_polarity', 'vader_compound', 'match']]

## 4. Confusion Matrix

In [ ]:
# Treat 'neutral' VADER predictions as 'negative' for binary comparison
df['vader_binary'] = df['vader_polarity'].map(
    lambda x: 'positive' if x == 'positive' else 'negative'
)
df['ground_truth'] = df['recommended'].map({True: 'positive', False: 'negative'})

from sklearn.metrics import confusion_matrix, classification_report
y_true = df['ground_truth']
y_pred = df['vader_binary']

cm = confusion_matrix(y_true, y_pred, labels=['positive', 'negative'])
print(classification_report(y_true, y_pred, labels=['positive', 'negative']))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Pos', 'Pred Neg'],
            yticklabels=['True Pos', 'True Neg'], ax=ax)
ax.set_title('VADER vs Ground Truth — Confusion Matrix')
fig.tight_layout()
fig.savefig(FIG / 'sent01_confusion_matrix.png', dpi=150)
plt.show()

## 5. Export to Parquet

Output schema for P17 (`steam-review-classifier`):

| Column | Type | Description |
|---|---|---|
| `app_name` | string | Game title |
| `review_text` | string | Raw review body |
| `recommended` | bool | Ground-truth label |
| `vader_polarity` | string | `positive` / `negative` / `neutral` |
| `vader_compound` | float | VADER compound score (−1 to 1) |

In [ ]:
out = df[['app_name', 'review_text', 'recommended', 'vader_polarity', 'vader_compound']].copy()

parquet_path = OUTPUTS / 'processed_reviews.parquet'
out.to_parquet(parquet_path, index=False, engine='pyarrow')

size_mb = parquet_path.stat().st_size / 1_048_576
print(f'Written: {parquet_path}  ({len(out):,} rows, {size_mb:.1f} MB)')
out.dtypes

## 6. Key Takeaways

- VADER achieves reasonable precision on *positive* reviews (short, enthusiastic text aligns with the VADER lexicon) but struggles with nuanced or sarcastic negative reviews.
- The 'neutral' bucket is sizeable (~15–20%) because Steam reviews often contain short or ambiguous text.
- VADER accuracy serves as a ceiling-floor baseline: P17's fine-tuned DistilBERT should substantially exceed it.
- The processed parquet at `outputs/processed_reviews.parquet` provides a clean, labelled dataset for P17.